# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [3]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [4]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## LLMChain

In [5]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [17]:
#Replace None by your own value and justify
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)


In [18]:
prompt = ChatPromptTemplate.from_template( #Write a query that would take a variable to describe any product
                                            "Write a description about {product}"
)

In [19]:

chain = LLMChain(llm=llm, prompt=prompt)

In [23]:
import numpy as np
product = df["Product"][np.random.randint(len(df))]
chain.run(product)

"A handheld milk frother is a versatile kitchen tool designed to create rich, creamy froth for your beverages with ease. Compact and lightweight, this battery-operated device typically features a stainless steel whisk head, which rapidly spins to incorporate air into milk, cream, or plant-based alternatives, transforming them into airy froth or silky microfoam. \n\nIdeal for coffee enthusiasts, a handheld milk frother is perfect for preparing lattes, cappuccinos, macchiatos, and even hot chocolate. With just a push of a button, you can achieve barista-quality results in the comfort of your home. Its user-friendly design allows for effortless operation and easy cleaning—simply rinse the whisk under warm water after use.\n\nMany models are designed with ergonomic handles for a comfortable grip and come in various colors and finishes to suit any kitchen aesthetic. Due to its compact size, a handheld milk frother can be easily stored in cabinets or drawers, making it a practical addition t

## SimpleSequentialChain

In [24]:
from langchain.chains import SimpleSequentialChain

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    #Repeat the initial query or create a new query that would feed into the second prompt
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [ ]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    #Write the second prompt query that takes an input variable whose input will come from the previous prompt"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [ ]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [ ]:
overall_simple_chain.run(product)

**Repeat the above twice for different products**

## SequentialChain

In [33]:
from langchain.chains import SequentialChain

In [28]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)


first_prompt = ChatPromptTemplate.from_template("Translate this {review} to french"
  #This prompt should translate a review
)

chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="french_text"
                    )


In [29]:
second_prompt = ChatPromptTemplate.from_template("Summarize this {review} in max 50 word"
    #Write a promplt to summarize a review
)

chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summarized_text"
                    )


In [37]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template("Translate {review} to german"
    
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="german_text"
                      )


In [43]:

# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template("Write a polite follow-up message to the customer, in German.\n\n"
    "Review (German translation): {german_text}\n"
    "Summary (50 words): {summarized_text}"
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [44]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["review"],
    output_variables=["german_text", "summarized_text"],
    verbose=True
)

In [47]:
review = df.Review[np.random.randint(len(df))]
overall_chain(review)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.


{'review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\r\nVieux lot ou contrefaçon !?",
 'german_text': 'Ich finde den Geschmack mittelmäßig. Der Schaum hält nicht, das ist seltsam. Ich kaufe die gleichen im Handel und der Geschmack ist viel besser... Alter Lot oder Fälschung!?',
 'summarized_text': "L'auteur trouve le goût médiocre et la mousse instable du produit, le comparant défavorablement à des alternatives commerciales. Il s'interroge sur la qualité, suggérant un vieux lot ou une contrefaçon."}

**Repeat the above twice for different products or reviews**

## Router Chain

In [48]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [49]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [50]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [51]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [52]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [53]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [55]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [56]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [57]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

C:\Users\PCACER\AppData\Local\Temp\ipykernel_12560\3038952769.py:1: LangChainDeprecationWarning: Use RunnableLambda to select from multiple prompt templates. See example in API reference: https://api.python.langchain.com/en/latest/chains/langchain.chains.router.multi_prompt.MultiPromptChain.html
  chain = MultiPromptChain(router_chain=router_chain,


In [58]:
chain.run("What is black body radiation?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


physics: {'input': 'What is black body radiation?'}
> Finished chain.


'Black body radiation refers to the electromagnetic radiation emitted by an idealized object known as a "black body." A black body is a perfect absorber of all incident radiation, meaning it absorbs all wavelengths of light and does not reflect any. When heated, a black body emits radiation in a characteristic spectrum that depends solely on its temperature.\n\nThe key features of black body radiation include:\n\n1. **Temperature Dependence**: The spectrum of radiation emitted by a black body is determined by its temperature. As the temperature increases, the peak wavelength of the emitted radiation shifts to shorter wavelengths (this is described by Wien\'s displacement law).\n\n2. **Planck\'s Law**: The distribution of emitted radiation across different wavelengths is described by Planck\'s law, which shows that the intensity of radiation increases with temperature and varies with wavelength.\n\n3. **Stefan-Boltzmann Law**: The total energy emitted per unit surface area of a black bo

In [59]:
chain.run("what is 2 + 2")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


math: {'input': 'what is 2 + 2'}
> Finished chain.


'To solve the problem \\(2 + 2\\), we can break it down into its component parts:\n\n1. Identify the numbers involved: We have the numbers 2 and 2.\n2. Understand the operation: The operation we are performing is addition.\n\nNow, we can combine the two numbers:\n\n\\[\n2 + 2 = 4\n\\]\n\nSo, the answer to the question \\(2 + 2\\) is \\(4\\).'

In [60]:
chain.run("Why does every cell in our body contain DNA?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA serves as the fundamental blueprint for life. Here are several key reasons why this is the case:\n\n1. **Genetic Information Storage**: DNA (deoxyribonucleic acid) contains the genetic instructions necessary for the development, functioning, growth, and reproduction of all living organisms. It encodes the information required to produce proteins, which perform a vast array of functions within the cell.\n\n2. **Cellular Function and Identity**: Each cell in our body has a specific role, whether it be a muscle cell, nerve cell, or skin cell. The DNA in each cell contains the same genetic information, but different genes are expressed in different cell types. This selective gene expression allows cells to specialize and perform their unique functions while still retaining the complete genetic code.\n\n3. **Replication and Division**: When cells divide, they must replicate their DNA so that each new cell receives an identical copy of the gen

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [61]:
chain.run("What is the difference between RAM and ROM?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


computer science: {'input': 'What is the difference between RAM and ROM in computer systems?'}
> Finished chain.


'RAM (Random Access Memory) and ROM (Read-Only Memory) are both types of memory used in computer systems, but they serve different purposes and have distinct characteristics. Here’s a breakdown of their differences:\n\n### 1. Definition:\n- **RAM (Random Access Memory)**: This is a type of volatile memory that temporarily stores data and machine code currently being used by the computer. It allows for quick read and write access to a storage medium.\n- **ROM (Read-Only Memory)**: This is a type of non-volatile memory that is used to store firmware or software that is not expected to change frequently. It retains its contents even when the power is turned off.\n\n### 2. Volatility:\n- **RAM**: Volatile memory, meaning it loses its contents when the power is turned off.\n- **ROM**: Non-volatile memory, meaning it retains its contents even when the power is turned off.\n\n### 3. Purpose:\n- **RAM**: Used for temporary storage of data that the CPU needs while performing tasks. It allows fo

In [62]:
chain.run("What was the color of the Soviet union flag?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


History: {'input': 'What was the color of the Soviet Union flag?'}
> Finished chain.


'The flag of the Soviet Union was primarily red. It featured a golden hammer and sickle, which symbolized the alliance between industrial workers and agricultural peasants, along with a gold-bordered red star above them. The red color of the flag was associated with the communist movement and represented revolution and the blood of workers who fought for their rights. The flag was officially adopted in 1923 and remained in use until the dissolution of the Soviet Union in 1991.'

In [63]:
chain.run("what is the sum of log 1 and log 2?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


math: {'input': 'What is the sum of log 1 and log 2?'}
> Finished chain.


'To find the sum of \\(\\log 1\\) and \\(\\log 2\\), we can break it down into its component parts:\n\n1. **Calculate \\(\\log 1\\)**:\n   - The logarithm of 1 in any base is always 0. This is because any number raised to the power of 0 equals 1. Therefore, \\(\\log 1 = 0\\).\n\n2. **Calculate \\(\\log 2\\)**:\n   - The value of \\(\\log 2\\) depends on the base of the logarithm. If we assume the base is 10 (common logarithm), then \\(\\log_{10} 2\\) is approximately 0.3010. If the base is \\(e\\) (natural logarithm), then \\(\\ln 2\\) is approximately 0.6931. However, for the purpose of this problem, we can denote \\(\\log 2\\) as simply \\(\\log 2\\).\n\n3. **Sum the results**:\n   - Now we can sum the two results:\n   \\[\n   \\log 1 + \\log 2 = 0 + \\log 2 = \\log 2\n   \\]\n\nThus, the sum of \\(\\log 1\\) and \\(\\log 2\\) is \\(\\log 2\\).'

In [64]:
chain.run("What is the biggest human organ?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


biology: {'input': 'What is the biggest human organ?'}
> Finished chain.


'The largest human organ is the skin. It serves as a protective barrier for the body, regulating temperature, and facilitating the sensation of touch, pain, and pressure. The skin is composed of multiple layers, including the epidermis, dermis, and subcutaneous tissue, and it plays a crucial role in maintaining homeostasis and protecting against environmental hazards. In adults, the skin can cover an area of about 1.5 to 2 square meters and can weigh around 3 to 4 kilograms (6.6 to 8.8 pounds), making it the most extensive organ in the human body.'

In [65]:
chain.run("I am learning AI engineering any tips?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


None: {'input': 'I am learning AI engineering. Do you have any tips?'}
> Finished chain.


"Absolutely! Here are some tips to help you on your journey to becoming an AI engineer:\n\n1. **Understand the Fundamentals**:\n   - **Mathematics**: Brush up on linear algebra, calculus, probability, and statistics. These are crucial for understanding algorithms and models.\n   - **Programming**: Proficiency in programming languages like Python is essential, as it is widely used in AI development. Familiarize yourself with libraries such as NumPy, Pandas, TensorFlow, and PyTorch.\n\n2. **Learn Machine Learning**:\n   - Start with the basics of machine learning, including supervised and unsupervised learning, classification, regression, clustering, and reinforcement learning.\n   - Explore online courses (like those from Coursera, edX, or Udacity) that cover machine learning concepts and practical applications.\n\n3. **Deep Learning**:\n   - Once you have a grasp of machine learning, delve into deep learning. Understand neural networks, convolutional neural networks (CNNs), recurrent n